<a href="https://colab.research.google.com/github/Ahmed-coder874/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ahmed-coder874/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

The unit of analysis is one content page.

Each row represents one content page with its historical search and performance information.

The analysis uses a historical 90-day performance window because the opportunity score is based on recent content performance signals such as impressions, clicks, CTR, and ranking position.

The goal is to identify pages that may have improvement opportunities based on observed historical patterns.


In [11]:
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

print("Rows:", len(df))
print("Columns:", len(df.columns))

df.head()


Rows: 30000
Columns: 44


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


## 2. Fields: feature / label / context / excluded

Feature fields:
- impressions_90d
- clicks_90d
- ctr
- avg_position
- search_volume
- content_type
- word_count

These fields are used as model inputs because they describe observed content performance.

Label:
- trend_direction

This is used as a proxy label because it represents observed historical change in content performance.

Context fields:
- content_id
- client_id (if available)
- page metadata fields

These fields help identify or group records but may not directly predict the outcome.

Excluded fields:
- Future performance fields
- Any manually created recommendation fields
- Fields generated after the prediction period

These are excluded to reduce leakage and avoid using information that would not be available at prediction time.


In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

## 2. Fields: feature / label / context / excluded

Feature fields:
- search_volume
- competition
- competition_level
- cpc
- content_type
- main_intent
- word_count
- char_count
- impressions_90d
- clicks_90d
- pageviews_90d
- sessions_90d
- ctr
- avg_position
- engagement_rate
- scroll_rate
- content_age_days
- days_since_last_update

These fields represent observed content and search performance signals used to understand optimization opportunities.

Label:
- trend_direction

This is a proxy label created from observed historical performance changes. It shows whether a page trend was up, down, or stable.

Context fields:
- content_id
- client_id
- provider_used
- model_used

These fields provide additional information about the record but are not direct performance signals.

Excluded fields:
- trend_pct

This field is excluded because it directly describes the size of the historical change and could leak information about the target.

Future performance fields are also excluded because they would not be available when making recommendations.


In [13]:
print("Number of rows:", len(df))

print("\nColumns:")
print(df.columns.tolist())

print("\nMissing values:")
print(df.isnull().sum().sort_values(ascending=False).head(10))

df.head()
print("Total rows:", len(df))
print("Unique content pages:", df["content_id"].nunique())

if len(df) == df["content_id"].nunique():
    print("Verified: one row represents one content page")
else:
    print("Some content pages have multiple rows")


Number of rows: 30000

Columns:
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']

Missing values:
provider_used        21438
word_count            7699
char_count            7699
word_count_tier       7699
char_count_tier       7699
model_used            5733
trend_pct         

## 4. Data limits

This dataset can show observed relationships between historical content signals and performance outcomes.

However, it cannot prove causation. For example, it cannot prove that changing a page will improve rankings.

The data may have limitations:
- Historical performance may not represent future search behavior.
- Some pages may have more history available than others.
- Search engine algorithm changes are not fully represented.
- Overlapping time windows can make some observations related.

The model should be used as decision support, not as a guarantee of improvement.


In [14]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

In [15]:
import os
import subprocess

REPO_URL = "https://github.com/Ahmed-coder874/flyrank-ml-internship"
REPO_DIR = "flyrank-ml-internship"

if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)

os.chdir(REPO_DIR)

print("Current folder:", os.getcwd())


Current folder: /content/flyrank-ml-internship/flyrank-ml-internship


In [16]:
import os

print(os.path.exists("data/raw/content_refresh_anonymized.csv"))


True
